In [23]:
import re

def remove_toc_and_index(text: str) -> str:
    # normalise les fins de ligne
    text = text.replace("\r\n", "\n").replace("\r", "\n")


    # supprime les artefacts de liens PDF du type [ } 12] ou [} 12]
    text = re.sub(r"\[\s*}\s*\d+\s*\]", "", text)

    # supprime toute la table des matières
    text = re.sub(
        r"## Table of contents.*?(?=\n## Overview of compensation parameters\b)",
        "",
        text,
        flags=re.DOTALL
    )

    # supprime tout le keyword index
    text = re.sub(
        r"## Keyword index.*?(?=\n## 4 Appendix\b)",
        "",
        text,
        flags=re.DOTALL
    )

    # supprimer les balises d'image <!-- image -->
    text = re.sub(r"<!-- image -->", "", text)

    # nettoie les lignes vides en trop
    text = re.sub(r"\n{3,}", "\n\n", text).strip()

    return text

In [10]:
from docling.document_converter import DocumentConverter
from docling.chunking import HybridChunker

FILE_PATH = "./../data/Manuals/mds_axis_compensation_en.pdf"
MODEL = "Qwen/Qwen3-Embedding-0.6B"
# 1. Conversion PDF -> DoclingDocument (pas de chunking ici)
converter = DocumentConverter()
pdf_doc = converter.convert(FILE_PATH).document
full_md = pdf_doc.export_to_markdown()

[INFO] 2026-04-21 16:42:10,735 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-04-21 16:42:10,739 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\crist\perso\master\2\pi\pi\.venv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-04-21 16:42:10,739 [RapidOCR] main.py:57: Using C:\Users\crist\perso\master\2\pi\pi\.venv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-04-21 16:42:10,809 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-04-21 16:42:10,811 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\crist\perso\master\2\pi\pi\.venv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-04-21 16:42:10,812 [RapidOCR] main.py:57: Using C:\Users\crist\perso\master\2\pi\pi\.venv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-04-21 16:42:10,845 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-0

In [25]:
from docling.datamodel.base_models import InputFormat

# 2) nettoyage markdown
cleaned_md = remove_toc_and_index(full_md)
cleaned_doc = converter.convert_string(
    cleaned_md,
    format=InputFormat.MD,
).document

In [ ]:
from docling_core.transforms.chunker.hierarchical_chunker import (
    ChunkingDocSerializer,
    ChunkingSerializerProvider,
)
from docling_core.transforms.serializer.markdown import MarkdownTableSerializer

class MDTableSerializerProvider(ChunkingSerializerProvider):
    def get_serializer(self, doc):
        return ChunkingDocSerializer(
            doc=doc,
            table_serializer=MarkdownTableSerializer(),
        )

# 3. Chunking sur le doc nettoyé
chunker = HybridChunker(
    tokenizer=MODEL,
    serializer_provider=MDTableSerializerProvider(),
    max_tokens=1500,
    merge_peers=True,
    repeat_table_header=True,
    omit_header_on_overflow=True,
)

raw_chunks = list(chunker.chunk(dl_doc=cleaned_doc))
chunks = [chunker.contextualize(chunk=chunk).strip() for chunk in raw_chunks]